In [1]:
import anatomist.api as anatomist
from soma.qt_gui.qtThread import QtThreadCall
from soma.qt_gui.qt_backend import Qt

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
from soma import aims
import pandas as pd
import numpy as np
import scipy.stats
import scipy
import json
import glob
import sys
import os

In [3]:
a = anatomist.Anatomist()

existing QApplication: 0
QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-ad279118'


global modules: /casa/host/build/share/anatomist-5.2/python_plugins
home   modules: /casa/home/.anatomist/python_plugins
create qapp
done
Starting Anatomist.....
config file : /casa/home/.anatomist/config/settings.cfg
PyAnatomist Module present
PythonLauncher::runModules()
loading module simple_controls
loading module save_resampled
loading module selection
loading module bsa_proba
loading module modelGraphs
loading module profilewindow
loading module ana_image_math
loading module paletteViewer
loading module foldsplit
loading module anacontrolmenu
loading module gradientpalette
loading module palettecontrols
loading module meshsplit
loading module volumepalettes
loading module gltf_io
loading module infowindow
loading module histogram
loading module measure
loading module statsplotwindow
loading module valuesplotwindow
all python modules loaded
Anatomist started.


In [4]:
Rspam_model = "/casa/host/build/share/brainvisa-share-5.2/models/models_2008/descriptive_models/segments/global_registered_spam_right/meshes/Rspam_model_meshes_1.arg"
Lspam_model = "/casa/host/build/share/brainvisa-share-5.2/models/models_2008/descriptive_models/segments/global_registered_spam_left/meshes/Lspam_model_meshes_1.arg"
json_regions = "/neurospin/dico/data/deep_folding/current/sulci_regions_gridsearch.json"

In [6]:
with open(json_regions) as f:
    regions = json.load(f)

In [7]:
path_to_Champollion = "/home/ad279118/tmp1"
folder = "NOPCA" #32PCs, NOPCA

# Setup for the report
with open(f"{path_to_Champollion}/list_model.txt") as f:
    regions_models = [line.strip() for line in f if line.strip() and not line.strip().startswith("#")]

regions_models.sort()
print(regions_models)

# Initialize an empty list to store the heritability data
title_heritability = "Maximum Heritability Estimates (h2) for Brain Regions"

heritability_data = []

for region_model in regions_models:
    region, model = region_model.split('/')
    base_path = os.path.expanduser(f"{path_to_Champollion}/{region}/{model}/{folder}/white.British.ancestry")
    h2_path = os.path.join(base_path, "h2_summary.tsv")
    
    if os.path.exists(h2_path):
        h2_df = pd.read_csv(h2_path, sep="\t")
        if 'pheno' in h2_df.columns and 'h2' in h2_df.columns:
            # Extract the dimension with the highest heritability
            max_h2_row = h2_df.loc[h2_df['h2'].idxmax()]
            heritability_data.append([region, max_h2_row['pheno'], max_h2_row['h2']])

# Create a DataFrame for the heritability data
heritability_df = pd.DataFrame(heritability_data, columns=['Region', 'Most Heritable Dimension', 'h2'])
heritability_df = heritability_df.sort_values(by='h2')

['CINGULATE_left/name17-24-32_191', 'CINGULATE_right/name17-24-32_237', 'FCLp-subsc-FCLa-INSULA_left/name17-43-58_232', 'FCLp-subsc-FCLa-INSULA_right/name17-47-16_166', 'FCMpost-SpC_left/name06-21-10_231', 'FCMpost-SpC_right/name06-34-24_229', 'FColl-SRh_left/name06-43-43_210', 'FColl-SRh_right/name06-56-15_113', 'FIP_left/name07-03-29_175', 'FIP_right/name06-17-01_29', 'FPO-SCu-ScCal_left/name07-13-21_118', 'FPO-SCu-ScCal_right/name07-15-26_174', 'LARGE_CINGULATE_left/name06-16-54_61', 'LARGE_CINGULATE_right/name07-22-35_179', 'Lobule_parietal_sup_left/name07-23-04_36', 'Lobule_parietal_sup_right/name07-24-01_193', 'OCCIPITAL_left/name07-34-40_229', 'OCCIPITAL_right/name07-38-28_182', 'SC-SPeC_left/name22-16-47_177', 'SC-SPeC_right/name06-17-00_24', 'SC-SPoC_left/name07-57-18_53', 'SC-SPoC_right/name07-58-03_243', 'SC-sylv_left/name07-58-00_111', 'SC-sylv_right/name06-17-02_84', 'SFinf-BROCA-SPeCinf_left/name08-00-45_128', 'SFinf-BROCA-SPeCinf_right/name08-00-44_234', 'SFint-FCMant_le

In [8]:
df = heritability_df.rename(columns={"Region":"region"})

In [9]:
res = df.groupby(['region']).mean()
res["side"] = res.index.str.split('_').str[-1]
res = res.reset_index()

In [10]:
def get_sulci(region):
    region = region.replace("CINGULATE", "CINGULATE.")
    region = region.replace("ORBITAL", "S.Or.")
    list_sulci = list(regions['brain'][f"{region}"].keys())
    list_sulci = [x.replace("paracingular.", "S.F.int.") for x in list_sulci]
    return list_sulci

dic = {
    "FCLp-subsc-FCLa-INSULA": "F.C.L.p.-subsc.-F.C.L.a.-INSULA.",
    "FCMpost-SpC": "F.C.M.post.-S.p.C.",
    "FColl-SRh": "F.Coll.-S.Rh.",
    "FIP": "F.I.P.",
    "FPO-SCu-ScCal": "F.P.O.-S.Cu.-Sc.Cal.",
    "fronto-parietal_medial_face": "fronto-parietal_medial_face.", 
    "Lobule_parietal_sup": "Lobule_parietal_sup.",
    "ORBITAL": "S.Or.",
    "SC-SPeC": "S.C.-S.Pe.C.",
    "SC-SPoC": "S.C.-S.Po.C.",
    "SC-sylv": "S.C.-sylv.",
    "SFinf-BROCA-SPeCinf": "S.F.inf.-BROCA-S.Pe.C.inf.",
    "SFint-FCMant": "S.F.int.-F.C.M.ant.",
    "SFint-SR": "S.F.int.-S.R.",
    "SFinter-SFsup": "S.F.inter.-S.F.sup.",
    "SFmarginal-SFinfant":"S.F.marginal-S.F.inf.ant.",
    "SFmedian-SFpoltr-SFsup": "S.F.median-S.F.pol.tr.-S.F.sup.",
    "SOr-SOlf": "S.Or.-S.Olf.",
    "SOr": "S.Or.",
    "SPeC": "S.Pe.C.",
    "SPoC": "S.Po.C.",
    "STi-SOTlat": "S.T.i.-S.O.T.lat.",
    "STi-STs-STpol": "S.T.i.-S.T.s.-S.T.pol.",
    "STsbr": "S.T.s.br.",
    "STs": "S.T.s.",
    "ScCal-SLi": "Sc.Cal.-S.Li.",
    "SsP-SPaint": "S.s.P.-S.Pa.int.",
}

for key in dic.keys():
    res["region"] = res["region"].str.replace(key, dic[key])
res.head()

,region,h2,side
0,CINGULATE_left,0.1467,left
1,CINGULATE_right,0.0995,right
2,F.C.L.p.-subsc.-F.C.L.a.-INSULA._left,0.3003,left
3,F.C.L.p.-subsc.-F.C.L.a.-INSULA._right,0.2912,right
4,F.C.M.post.-S.p.C._left,0.1568,left


In [11]:
res['sulcus'] = res.apply(lambda x: get_sulci(x.region), axis=1)
res.head()

,region,h2,side,sulcus
0,CINGULATE_left,0.1467,left,"[S.F.int._left, F.C.M.ant._left]"
1,CINGULATE_right,0.0995,right,"[S.F.int._right, F.C.M.ant._right]"
2,F.C.L.p.-subsc.-F.C.L.a.-INSULA._left,0.3003,left,"[F.C.L.p._left, F.C.L.r.sc.ant._left, F.C.L.r...."
3,F.C.L.p.-subsc.-F.C.L.a.-INSULA._right,0.2912,right,"[F.C.L.p._right, F.C.L.r.sc.ant._right, F.C.L...."
4,F.C.M.post.-S.p.C._left,0.1568,left,"[F.C.M.post._left, S.C.LPC._left, S.p.C._left]"


In [12]:
res = res.sort_values(by="h2", ascending=False)
res = res.explode("sulcus")
res[res.region.str.contains("S.T.s.")]

,region,h2,side,sulcus
47,S.T.i.-S.T.s.-S.T.pol._right,0.2466,right,S.T.i.ant._right
47,S.T.i.-S.T.s.-S.T.pol._right,0.2466,right,S.T.i.post._right
47,S.T.i.-S.T.s.-S.T.pol._right,0.2466,right,S.T.s._right
47,S.T.i.-S.T.s.-S.T.pol._right,0.2466,right,S.T.pol._right
46,S.T.i.-S.T.s.-S.T.pol._left,0.2339,left,S.T.i.ant._left
46,S.T.i.-S.T.s.-S.T.pol._left,0.2339,left,S.T.i.post._left
46,S.T.i.-S.T.s.-S.T.pol._left,0.2339,left,S.T.s._left
46,S.T.i.-S.T.s.-S.T.pol._left,0.2339,left,S.T.pol._left
49,S.T.s._right,0.2141,right,S.T.s._right
48,S.T.s._left,0.2073,left,S.T.s._left


In [13]:
def set_color_property(res, side):
    global dic

    if side == "L":
        spam_model_file = Lspam_model
    else:
        spam_model_file = Rspam_model
        
    dic[f"aims{side}"] = aims.read(spam_model_file)

    for vertex in dic[f"aims{side}"].vertices():
        vertex['h2'] = 0.

    for _, row in res.iterrows():
        for vertex in dic[f"aims{side}"].vertices():
            vname = vertex.get('name')
            if vname == row.sulcus:
                    vertex['h2'] = max(vertex['h2'],
                                                row.h2)
                    # if row.p < -np.log10(0.05/56):
                    #     vertex['p_value'] = 0.
                    # elif vertex['p_value'] != 0.: 
                    #     vertex['p_value'] = min(vertex['p_value'],
                    #                             row.p)
                    # else:
                    #     vertex['p_value'] = row.p
    
    dic[f"ana{side}"] = a.toAObject(dic[f"aims{side}"])

    dic[f"ana{side}"].setColorMode(dic[f"ana{side}"].PropertyMap)
    dic[f"ana{side}"].setColorProperty('h2')
    dic[f"ana{side}"].notifyObservers()
    
                
def visualize_whole_hemisphere(view_quaternion, side, i):
    global block
    global dic
    try:
        block
    except NameError:
        block = a.createWindowsBlock(4)

    dic[f"win{i}"] = a.createWindow('3D',
                                    block=block,
                                    no_decoration=True,
                                    options={'hidden': 1})
    dic[f"win{i}"].addObjects(dic[f"ana{side}"])
    dic[f"ana{side}"].setPalette("green_yellow_red",
                              minVal=0.05, maxVal=0.4,
                              absoluteMode=True)
    
    dic[f"win{i}"].camera(view_quaternion=view_quaternion)

    # 0;1;0.579487;1;0.992308;1#0;1;0.246154;0.822222;0.630769;0.311111;1;0#0;1;0.320513;0.0888889;1;0#0.5;1

middle_view = [0.5, -0.5, -0.5, 0.5]
side_view = [0.5, 0.5, 0.5, 0.5]
bottom_view = [0, -1, 0, 0]
top_view = [0, 0, 0, -1]
 
def visualize_whole(res, side, start):
    set_color_property(res, side)
    visualize_whole_hemisphere(middle_view if side == "L" else side_view, side, start+0)
    visualize_whole_hemisphere(top_view, side, start+1)
    visualize_whole_hemisphere(bottom_view, side, start+2)
    visualize_whole_hemisphere(side_view if side == "L" else middle_view, side, start+3)

In [14]:
dic = {}

visualize_whole(res, "L", 0)
visualize_whole(res, "R", 4)

Reading FGraph version 2.0
bounding box found : -90, -80, -90
                     90, 120, 60
Reading FGraph version 2.0
bounding box found : -90, -80, -90
                     90, 120, 60


Multitexturing present
function glActiveTexture found.
function glClientActiveTexture found.
function glBlendEquation found.
function glTexImage3D found.
function glMultiTexCoord3f found.
function glBindFramebuffer found.
function glBindRenderbuffer found.
function glFramebufferTexture2D found.
function glGenFramebuffers found.
function glGenRenderbuffers found.
function glFramebufferRenderbuffer found.
function glRenderbufferStorage found.
function glCheckFramebufferStatus found.
function glDeleteRenderbuffers found.
function glDeleteFramebuffers found.
Number of texture units: 4
function glUniform1f found.
function glUniform1i found.
function glUniform4fv found.
function glGetUniformLocation found.
function glMultTransposeMatrixf found.
function glAttachShader found.
function glDetachShader found.
function glCompileShader found.
function glCreateProgram found.
function glCreateShader found.
function glDeleteProgram found.
function glDeleteShader found.
function glGetProgramiv found.


QLayout: Attempting to add QLayout "" to QWidget "", which already has a layout


Position : 0.669287, 36.3165, -37.9376, 0
Position : 0.669287, 36.3165, -37.9376, 0
Position : -0.0391132, -14.7612, 16.4506, 0


: 